In [1]:
from model import clean_data_and_remove_outliers, predict_goals
import pandas as pd
from utils import create_attacking_indices
import warnings
import numpy as np
warnings.filterwarnings("ignore")

1. Download from kaggle

In [2]:
import kagglehub
import os
import pandas as pd

current_dir = os.getcwd()
os.chdir(current_dir)

path = kagglehub.dataset_download("hubertsidorowicz/football-players-stats-2024-2025")
csv_path = os.path.join(path, "players_data-2024_2025.csv")  # replace with the actual filename

df = pd.read_csv(csv_path)

In [9]:
df[['Player', 'Nation', 'Pos', 'Squad', 'Gls', 'xG']].sort_values(by='xG', ascending=False).head(10)

,Player,Nation,Pos,Squad,Gls,xG
1483,Robert Lewandowski,pl POL,FW,Barcelona,27,27.1
1691,Kylian Mbappé,fr FRA,FW,Real Madrid,31,25.9
2304,Mohamed Salah,eg EGY,FW,Liverpool,29,25.2
1093,Serhou Guirassy,gn GUI,FW,Dortmund,21,22.7
1109,Erling Haaland,no NOR,FW,Manchester City,22,22.0
823,Hugo Ekitike,fr FRA,FW,Eint Frankfurt,15,21.6
1219,Alexander Isak,se SWE,FW,Newcastle Utd,23,20.3
1317,Harry Kane,eng ENG,FW,Bayern Munich,26,20.3
1330,Moise Kean,it ITA,FW,Fiorentina,17,19.4
2177,Raphinha,br BRA,"FW,MF",Barcelona,18,19.2


In [4]:
cols=df.columns.to_list()
cols

['Rk',
 'Player',
 'Nation',
 'Pos',
 'Squad',
 'Comp',
 'Age',
 'Born',
 'MP',
 'Starts',
 'Min',
 '90s',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'CrdY',
 'CrdR',
 'xG',
 'npxG',
 'xAG',
 'npxG+xAG',
 'PrgC',
 'PrgP',
 'PrgR',
 'G+A-PK',
 'xG+xAG',
 'Rk_stats_shooting',
 'Nation_stats_shooting',
 'Pos_stats_shooting',
 'Comp_stats_shooting',
 'Age_stats_shooting',
 'Born_stats_shooting',
 '90s_stats_shooting',
 'Gls_stats_shooting',
 'Sh',
 'SoT',
 'SoT%',
 'Sh/90',
 'SoT/90',
 'G/Sh',
 'G/SoT',
 'Dist',
 'FK',
 'PK_stats_shooting',
 'PKatt_stats_shooting',
 'xG_stats_shooting',
 'npxG_stats_shooting',
 'npxG/Sh',
 'G-xG',
 'np:G-xG',
 'Rk_stats_passing',
 'Nation_stats_passing',
 'Pos_stats_passing',
 'Comp_stats_passing',
 'Age_stats_passing',
 'Born_stats_passing',
 '90s_stats_passing',
 'Cmp',
 'Att',
 'Cmp%',
 'TotDist',
 'PrgDist',
 'Ast_stats_passing',
 'xAG_stats_passing',
 'xA',
 'A-xAG',
 'KP',
 '1/3',
 'PPA',
 'CrsPA',
 'PrgP_stats_passing',
 'Rk_stats_passing_ty

2. Cleaning the data

In [5]:
df.shape

(2854, 267)

In [10]:
df_clean = clean_data_and_remove_outliers(df, z_threshold=3)

Removed 0 duplicate rows

Original dataset size: 2854
Cleaned dataset size: 2734
Removed 120 rows (4.20%)


3. Model predictions

In [7]:
results = predict_goals(df_clean)

Using 8 features for prediction

=== Model Performance ===
Train MAE: 0.239
Test MAE: 0.627
Train R²: 0.957
Test R²: 0.718

=== Top 10 Most Important Features ===
feature  importance
    SoT    0.742713
   PrgC    0.050972
  Sh/90    0.042729
     Sh    0.036549
    90s    0.036182
   SoT%    0.032340
 SoT/90    0.029986
Att 3rd    0.028529


3. Predictions

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
test_size=0.2
random_state=42

In [11]:
feature_cols = ['xG', 'npxG', 'Sh', 'SoT', '90s', 'Sh/90', 'SoT/90', 'SoT%', 'Att 3rd', 'PrgC']
target_col = 'Gls'    

X = df[feature_cols].copy()
y = df[target_col].copy()
X = X.fillna(X.median())

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state
    )

In [26]:
X_test.head()

,xG,npxG,Sh,SoT,90s,Sh/90,SoT/90,SoT%,Att 3rd,PrgC
1580,0.3,0.3,7,0,23.9,0.29,0.00,0.0,3,11
1742,0.5,0.5,3,2,3.5,0.86,0.58,66.7,2,2
772,4.1,4.1,31,9,28.5,1.09,0.32,29.0,7,42
1744,0.6,0.6,10,3,10.4,0.96,0.29,30.0,0,7
387,0.1,0.1,4,1,11.9,0.33,0.08,25.0,1,13


In [34]:
X_test.iloc[2]

xG          4.10
npxG        4.10
Sh         31.00
SoT         9.00
90s        28.50
Sh/90       1.09
SoT/90      0.32
SoT%       29.00
Att 3rd     7.00
PrgC       42.00
Name: 772, dtype: float64

In [35]:
df[['Player', 'xG', 'npxG', 'Sh', 'SoT', '90s', 'Sh/90', 'SoT/90', 'SoT%', 'Att 3rd','PrgC', 'Gls']].iloc[772]

Player     Abdoulaye Doucouré
xG                        4.1
npxG                      4.1
Sh                         31
SoT                         9
90s                      28.5
Sh/90                    1.09
SoT/90                   0.32
SoT%                     29.0
Att 3rd                     7
PrgC                       42
Gls                         3
Name: 772, dtype: object

In [ ]:
y_pred_test = results['model'].predict(X_test)

In [37]:
y_pred_test[2]

np.float64(4.24)

In [ ]:
new_data=df[df['Player']=='Kylian Mbappé']

In [8]:
new_predictions = results['model'].predict(new_data)

NameError: name 'new_data' is not defined

In [30]:
df['Touches in Att Pen / Att 3rd']

KeyError: 'Touches in Att Pen / Att 3rd'

In [12]:
exclude_patterns = [
    'Rk', 'Player', 'Nation', 'Squad', 'Comp',  # Identification columns
    'Gls', 'G+A', 'G-PK', 'G-xG', 'G/Sh', 'G/SoT',  # Goal-related (target leakage)
    'np:G-xG', 'xG+xAG', 'G+A-PK',  # More goal derivatives
    'onG', 'Born'  # Other non-predictive columns
]

# Get numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Filter out excluded columns (including partial matches)
feature_cols = [col for col in numeric_cols 
                if not any(pattern in col for pattern in exclude_patterns)]

print(f"Using {len(feature_cols)} features for prediction")
print(f"Excluded {len(numeric_cols) - len(feature_cols)} columns to avoid data leakage")

Using 194 features for prediction
Excluded 38 columns to avoid data leakage


In [13]:
target_col='Gls'

In [14]:
X = df[feature_cols].copy()
y = df[target_col].copy()

In [20]:
X[['CS%',
 'PKatt_stats_keeper',
 'PKA',
 'PKsv',
 'PKm',
 'PSxG',
 'PSxG+/-',
 '/90',
 'Att (GK)',
 'Thr',
 'Launch%',
 'AvgLen',
 'Opp',
 'Stp',
 'Stp%',
 '#OPA',
 '#OPA/90',
 'AvgDist']]

,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,PSxG,PSxG+/-,/90,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
new_data

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
1691,1692,Kylian Mbappé,fr FRA,FW,Real Madrid,es La Liga,25.0,1998.0,34,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Find columns that are completely empty (all NaN)
all_nan_columns = new_data.columns[new_data.isna().all()].tolist()
print(f"Columns with all NaN values: {all_nan_columns}")
print(f"Total: {len(all_nan_columns)} columns")

Columns with all NaN values: ['Mn/Sub', 'Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper', 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm', 'Rk_stats_keeper_adv', 'Nation_stats_keeper_adv', 'Pos_stats_keeper_adv', 'Comp_stats_keeper_adv', 'Age_stats_keeper_adv', 'Born_stats_keeper_adv', '90s_stats_keeper_adv', 'GA_stats_keeper_adv', 'PKA_stats_keeper_adv', 'FK_stats_keeper_adv', 'CK_stats_keeper_adv', 'OG_stats_keeper_adv', 'PSxG', 'PSxG/SoT', 'PSxG+/-', '/90', 'Cmp_stats_keeper_adv', 'Att_stats_keeper_adv', 'Cmp%_stats_keeper_adv', 'Att (GK)', 'Thr', 'Launch%', 'AvgLen', 'Opp', 'Stp', 'Stp%', '#OPA', '#OPA/90', 'AvgDist']
Total: 54 columns


In [27]:
keeper=df[df['Pos'].str.contains('GK', na=False)]

In [29]:
keeper[['Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper', 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm', 'Rk_stats_keeper_adv', 'Nation_stats_keeper_adv', 'Pos_stats_keeper_adv', 'Comp_stats_keeper_adv', 'Age_stats_keeper_adv', 'Born_stats_keeper_adv', '90s_stats_keeper_adv', 'GA_stats_keeper_adv', 'PKA_stats_keeper_adv', 'FK_stats_keeper_adv', 'CK_stats_keeper_adv', 'OG_stats_keeper_adv', 'PSxG', 'PSxG/SoT', 'PSxG+/-', '/90', 'Cmp_stats_keeper_adv', 'Att_stats_keeper_adv', 'Cmp%_stats_keeper_adv', 'Att (GK)', 'Thr', 'Launch%', 'AvgLen', 'Opp', 'Stp', 'Stp%', '#OPA', '#OPA/90', 'AvgDist']]

,Rk_stats_keeper,Nation_stats_keeper,Pos_stats_keeper,Comp_stats_keeper,Age_stats_keeper,Born_stats_keeper,MP_stats_keeper,Starts_stats_keeper,Min_stats_keeper,90s_stats_keeper,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
33,1.0,es ESP,GK,es La Liga,37.0,1987.0,19.0,19.0,1710.0,19.0,...,507.0,91.0,28.0,31.2,257.0,14.0,5.4,16.0,0.84,11.7
42,2.0,es ESP,GK,es La Liga,23.0,2000.0,14.0,14.0,1206.0,13.4,...,341.0,51.0,37.5,33.9,143.0,20.0,14.0,17.0,1.27,14.2
82,3.0,de GER,GK,de Bundesliga,23.0,2000.0,2.0,1.0,91.0,1.0,...,37.0,5.0,59.5,44.7,15.0,0.0,0.0,0.0,0.00,NaN
87,4.0,br BRA,GK,eng Premier League,31.0,1992.0,28.0,28.0,2508.0,27.9,...,877.0,128.0,19.7,26.6,255.0,11.0,4.3,49.0,1.76,16.0
147,5.0,fr FRA,GK,eng Premier League,31.0,1993.0,26.0,25.0,2260.0,25.1,...,788.0,119.0,34.4,32.1,394.0,15.0,3.8,30.0,1.19,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2745,208.0,wls WAL,GK,eng Premier League,31.0,1993.0,2.0,1.0,135.0,1.5,...,46.0,9.0,37.0,32.0,11.0,0.0,0.0,2.0,1.33,20.7
2754,209.0,de GER,GK,de Bundesliga,25.0,1999.0,25.0,25.0,2250.0,25.0,...,789.0,116.0,40.2,34.5,491.0,27.0,5.5,18.0,0.72,11.1
2828,210.0,de GER,GK,de Bundesliga,29.0,1994.0,32.0,32.0,2880.0,32.0,...,977.0,132.0,46.2,38.7,524.0,45.0,8.6,58.0,1.81,14.0
2835,211.0,de GER,GK,de Bundesliga,29.0,1995.0,34.0,34.0,3060.0,34.0,...,1204.0,188.0,27.7,31.9,464.0,35.0,7.5,34.0,1.00,13.5
